In [1]:
import scipy.stats as stats
import numpy as np
import csv

In [2]:
### FUNCTION LIBRARY ###
def read_csv_file(file_path):
    values = []
    with open(file_path, 'r') as file:
        reader = csv.reader(file)
        for row in reader:
            values.extend(row)
    return values

def hoeffd_example(X, Y):
    # New dataset of 10 data points representing pairs of (height_in_inches, weight_in_pounds), with the last 3 being ties:

    # The 'average' ranking method assigns the average of the ranks that would have been assigned to all the tied values.
    R = stats.rankdata(X, method='average')  # Rank of MLD
    S = stats.rankdata(Y, method='average')  # Rank of RH

    N = len(X)  # Total number of data points

    # Q is an array that will hold a special sum for each data point, which is crucial for Hoeffding's D computation.
    Q = np.zeros(N)
    # Loop through each data point to calculate its Q value.
    for i in range(N):
        # For each data point 'i', count how many points have both a lower height and weight rank (concordant pairs).
        Q[i] = 1 + sum(np.logical_and(R < R[i], S < S[i]))

        # Adjust Q[i] for ties: when both ranks are equal, it contributes partially (1/4) to the Q[i] value.
        # The "- 1" accounts for not including the point itself in its own comparison.
        Q[i] += (1/4) * (sum(np.logical_and(R == R[i], S == S[i])) - 1)

        # When only the height rank is tied but the weight rank is lower, it contributes half (1/2) to the Q[i] value.
        Q[i] += (1/2) * sum(np.logical_and(R == R[i], S < S[i]))

        # Similarly, when only the weight rank is tied but the height rank is lower, it also contributes half (1/2).
        Q[i] += (1/2) * sum(np.logical_and(R < R[i], S == S[i]))

    # Calculate intermediate sums required for Hoeffding's D formula:

    # D1: This sum leverages the Q values calculated earlier. Each Q value encapsulates information about how
    # a data point's ranks relate to others in both sequences, including concordance and adjustments for ties.
    # The term (Q - 1) * (Q - 2) for each data point quantifies the extent to which the ranks of this point
    # are concordant with others, adjusted for the expected concordance under independence.
    # Summing these terms across all data points (D1) aggregates this concordance information for the entire dataset.
    D1 = sum((Q - 1) * (Q - 2))

    # D2: This sum involves products of rank differences for each sequence, adjusted for ties. The term
    # (R - 1) * (R - 2) * (S - 1) * (S - 2) for each data point captures the interaction between the rank variances
    # within each sequence, providing a measure of how the joint rank distribution diverges from what would
    # be expected under independence due to the variability in ranks alone, without considering their pairing.
    # Summing these products across all data points (D2) gives a global assessment of this divergence.
    D2 = sum((R - 1) * (R - 2) * (S - 1) * (S - 2))

    # D3: This sum represents an interaction term that combines the insights from Q values with rank differences.
    # The term (R - 2) * (S - 2) * (Q - 1) for each data point considers the rank variances alongside the Q value,
    # capturing how individual data points' rank concordance/discordance contributes to the overall dependency measure,
    # adjusted for the expected values under independence. Summing these terms (D3) integrates these individual
    # contributions into a comprehensive interaction term for the dataset.
    D3 = sum((R - 2) * (S - 2) * (Q - 1))

    # The final computation of Hoeffding's D integrates D1, D2, and D3, along with normalization factors
    # that account for the sample size (N). The normalization ensures that Hoeffding's D is scaled appropriately,
    # allowing for meaningful comparison across datasets of different sizes. The formula incorporates these sums
    # and normalization factors in a way that balances the contributions of concordance, discordance, and rank variances,
    # resulting in a statistic that robustly measures the degree of association between the two sequences.
    D = 30 * ((N - 2) * (N - 3) * D1 + D2 - 2 * (N - 2) * D3) / (N * (N - 1) * (N - 2) * (N - 3) * (N - 4))

    # Return the computed Hoeffding's D value.
    return D

## GRUAN Correlation test

In [3]:
# Import GRUAN data
mld_file_path = '/home/chinahg/GCresearch/contrailuncertainty/GRUAN_processing/GRUAN_MLD.csv'
rh_file_path = '/home/chinahg/GCresearch/contrailuncertainty/GRUAN_processing/GRUAN_RH.csv'

# Call the function to read the CSV file
GRUAN_MLD = np.array(read_csv_file(mld_file_path), dtype = float)
GRUAN_RH_raw = np.array(read_csv_file(rh_file_path))

In [4]:

"""
This code snippet deletes values from the `GRUAN_MLD` array based on a condition applied to the `GRUAN_RH` array.

Summary:
- The code initializes an empty list `indices_to_delete` to store the indices of values that need to be deleted.
- It then iterates over each element in the `GRUAN_RH` array and converts it to a float value.
- If the converted value is less than 100, the index of that value is appended to the `indices_to_delete` list.
- Finally, the corresponding values in the `GRUAN_MLD` array are deleted using the indices stored in `indices_to_delete`.
"""

indices_to_delete = []
GRUAN_RH = np.zeros(len(GRUAN_RH_raw))
for i in range(len(GRUAN_RH)):
    GRUAN_RH[i] = float(GRUAN_RH_raw[i])
    if GRUAN_RH[i] < 100:
        np.delete(GRUAN_RH, i)
        indices_to_delete.append(i)

# Delete corresponding values in GRUAN_MLD
GRUAN_MLD = np.delete(GRUAN_MLD, indices_to_delete)


In [5]:
len(GRUAN_MLD), len(GRUAN_RH)

(5637, 14318)

In [6]:
print("Spearman Coefficient: ",stats.spearmanr(GRUAN_MLD, GRUAN_RH))
print("Pearson Coefficient: ", stats.pearsonr(GRUAN_MLD, GRUAN_RH))

ValueError: all the input array dimensions for the concatenation axis must match exactly, but along dimension 0, the array at index 0 has size 5637 and the array at index 1 has size 14318

In [14]:
print("Hoeffding Coefficient: ", hoeffd_example(GRUAN_MLD[0:1000], GRUAN_RH[0:1000]))

Hoeffding Coefficient:  -0.00021719614007039578


## ERA5 Correlation Test

In [12]:
# Import ERA5 data
mld_file_path = '/home/chinahg/GCresearch/contrailuncertainty/ERA5_processing/ERA5_MLD.csv'
rh_file_path = '/home/chinahg/GCresearch/contrailuncertainty/ERA5_processing/ERA5_RH.csv'

# Call the function to read the CSV file
ERA5_MLD = np.array(read_csv_file(mld_file_path), dtype = float)
ERA5_cruiseRH = np.array(read_csv_file(rh_file_path), dtype = float)

In [ ]:
print("Spearman Coefficient: ",stats.spearmanr(ERA5_MLD, ERA5_cruiseRH))
print("Pearson Coefficient: ", stats.pearsonr(ERA5_MLD, ERA5_cruiseRH))

Spearman Coefficient:  SignificanceResult(statistic=0.06630863706539866, pvalue=0.0014635076993811803)
Pearson Coefficient:  PearsonRResult(statistic=0.04371822608606407, pvalue=0.03603652610489714)


In [13]:
print("Hoeffding Coefficient: ", hoeffd_example(ERA5_MLD, ERA5_cruiseRH,))

Hoeffding Coefficient:  -0.0005253938802506948
